# Food Demand Analysis

## Project Goal
Analyze food-demand data to understand order patterns, demand trends, popular items, and business insights.

This notebook is designed as a clean GitHub portfolio notebook.


In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)

# Put your dataset inside the repository's data/ folder.
# Rename the file here if your dataset has a different name.
DATA_PATH = Path("data/food_demand.csv")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Dataset not found. Add your CSV inside data/ and update DATA_PATH if needed."
    )

df = pd.read_csv(DATA_PATH)
print("Dataset shape:", df.shape)
df.head()


## 1. Data Understanding


In [ ]:
print("Columns:")
for i, col in enumerate(df.columns, 1):
    print(f"{i}. {col}")

print("\nData types:")
display(df.dtypes.to_frame("dtype"))

print("\nDuplicate rows:", df.duplicated().sum())


In [ ]:
missing = (
    df.isna().sum()
      .to_frame("missing_count")
      .assign(missing_pct=lambda x: (x["missing_count"] / len(df) * 100).round(2))
      .sort_values("missing_pct", ascending=False)
)
display(missing.head(20))


## 2. Basic Cleaning


In [ ]:
clean = df.copy()

# Remove duplicate rows
clean = clean.drop_duplicates()

# Trim whitespace in text columns
for col in clean.select_dtypes(include="object").columns:
    clean[col] = clean[col].astype(str).str.strip()

print("Cleaned shape:", clean.shape)


## 3. Demand Overview


In [ ]:
# Numeric summary
display(clean.describe(include="number").T)

# Categorical summary
cat_cols = clean.select_dtypes(include="object").columns.tolist()
for col in cat_cols[:5]:
    print(f"\nTop values in {col}:")
    display(clean[col].value_counts().head(10).to_frame("count"))


## 4. Trend Analysis


In [ ]:
# Try to detect a date column automatically
date_candidates = [c for c in clean.columns if "date" in c.lower() or "time" in c.lower()]
print("Possible date columns:", date_candidates)

if date_candidates:
    date_col = date_candidates[0]
    clean[date_col] = pd.to_datetime(clean[date_col], errors="coerce")
    clean = clean.sort_values(date_col)
    print("Using date column:", date_col)


In [ ]:
# Try to detect an order/demand/quantity column
qty_candidates = [
    c for c in clean.columns
    if any(k in c.lower() for k in ["quantity", "demand", "orders", "order_count", "num_orders"])
]
print("Possible demand columns:", qty_candidates)

if date_candidates and qty_candidates:
    date_col = date_candidates[0]
    qty_col = qty_candidates[0]

    trend = (
        clean.dropna(subset=[date_col])
             .set_index(date_col)[qty_col]
             .resample("M")
             .sum()
    )

    plt.figure(figsize=(10,4))
    plt.plot(trend.index, trend.values)
    plt.title("Monthly Food Demand Trend")
    plt.xlabel("Month")
    plt.ylabel(qty_col)
    plt.tight_layout()
    plt.show()


## 5. Popular Items / Categories


In [ ]:
item_candidates = [
    c for c in clean.columns
    if any(k in c.lower() for k in ["item", "product", "food", "category", "meal"])
]
print("Possible item/category columns:", item_candidates)

if item_candidates:
    item_col = item_candidates[0]

    if qty_candidates:
        qty_col = qty_candidates[0]
        top_items = (
            clean.groupby(item_col)[qty_col]
                 .sum()
                 .sort_values(ascending=False)
                 .head(10)
        )
    else:
        top_items = clean[item_col].value_counts().head(10)

    display(top_items.to_frame("demand"))

    plt.figure(figsize=(9,5))
    plt.barh(top_items.sort_values().index.astype(str), top_items.sort_values().values)
    plt.title("Top Food Items / Categories")
    plt.xlabel("Demand")
    plt.tight_layout()
    plt.show()


## 6. Business Insights


In [ ]:
print("Use the executed results above to write evidence-based insights such as:")
print("- Which items have the highest demand?")
print("- Which months or periods have peak demand?")
print("- Are there low-demand items that may cause excess inventory?")
print("- Which categories need better stock planning?")
print("- What actions could reduce food wastage or stock-outs?")


## 7. Conclusion

After running the notebook with the real dataset, replace this section with 4–6 actual findings and recommendations.

Examples:
- Improve stock planning for consistently high-demand items.
- Review low-demand products to reduce excess inventory.
- Use demand trends to plan procurement before peak periods.
